# 05 — Advanced analytics from saved artifacts (no GPU needed)

Everything here reads the experiment directories: comparisons across experiments, custom figures, and the low-level API on saved embeddings.

In [ ]:
from pathlib import Path

import polars as pl

from nflbowl import Experiment

ROOT = Path("/content/drive/MyDrive/NFLBowl/experiments")      # or a local copy of the exported zip
exp = Experiment.open(ROOT / "f4_main")
exp.list_artifacts().select("kind", "key", "path", "created_at")

In [ ]:
# compare completed experiments (e.g. two seeds) without recomputation
Experiment.compare_experiments([ROOT / "f4_main", ROOT / "f4_seed2"], level="full").select(
    "experiment", "representation", "cross_auc", "cross_auc_lo", "cross_auc_hi", "null_mean")

In [ ]:
# re-evaluate a saved embedding table with the low-level API (no Experiment state needed beyond the bundle)
import re

from nflbowl.evaluation import cross_domain_retrieval

df = exp.load_embeddings("F4")
E = df.select([c for c in df.columns if re.fullmatch(r"e\d+", c)]).to_numpy()
cross_domain_retrieval(E, df, n_boot=200, seed=0, combine_kinds=["forty"])      # forty-only Combine queries

In [ ]:
# a custom figure from saved tables, saved with its data next to the standard ones
from nflbowl.visualization import forest

m = exp.metrics(include_quick=False)
fig = forest(m, "representation", "within_auc", None, None, "Within-domain retrieval (repeat Combine attempts)", "AUC", ref=0.5, ref_label="chance")
exp.store.save_figure(fig, "custom_within_auc", source={"metrics": m})
fig